In [31]:
import os
import pandas as pd
import numpy as np
import gseapy as gp
import matplotlib.pyplot as plt
import networkx as nx

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

input_dir = "/home/ajarrah/PhD_Thesis/gene_paper/results_hippocampus"
output_dir = "/home/ajarrah/PhD_Thesis/gene_paper/GSEA_results_hippocampus"

os.makedirs(output_dir, exist_ok=True)


# Files

In [32]:
files = [
    "DE_AAD_vs_AC_Aged_AD_vs_Aged_Control.csv",
    "DE_AAD_vs_YAD_Aged_AD_vs_Young_AD.csv",
    "DE_AC_vs_YC_Aged_Control_vs_Young_Control_(aging_effect).csv",
    "DE_AD_vs_Control_All_AD_vs_All_Control.csv",
    "DE_Aged_vs_Young_All_Aged_vs_All_Young.csv",
    "DE_YAD_vs_YC_Young_AD_vs_Young_Control.csv"
]

# Pathway databases

In [33]:
gene_sets = {

    # Broad biological programs
    "Hallmark":
        "MSigDB_Hallmark_2020",

    # Cellular processes
    "GO_BP":
        "GO_Biological_Process_2023",

    # Curated signaling pathways
    "Reactome":
        "Reactome_2022",

    # Metabolic/signaling pathways
    "KEGG":
        "KEGG_2021_Mouse",

    # Brain-related pathways
    "WikiPathways":
        "WikiPathways_2024_Mouse",

    # Disease-focused
    "DisGeNET":
        "DisGeNET_2024"
}

""" 
extend the gene_sets dictionary with additional databases as needed, for example:
"GO_Molecular_Function_2023",
    "GO_Cellular_Component_2023",
    "BioPlanet_2019",
"""

' \nextend the gene_sets dictionary with additional databases as needed, for example:\n"GO_Molecular_Function_2023",\n    "GO_Cellular_Component_2023",\n    "BioPlanet_2019",\n'

# Create ranking

In [34]:
def create_rank_file(df):

    df = df.copy()

    # remove missing values
    df = df.dropna(subset=[
        "gene",
        "log2FC",
        "padj",
        "pval"
    ])

    # remove duplicated genes
    df = df.drop_duplicates( subset="gene", keep="first")

    # avoid log(0)
    df["padj"] = df["padj"].clip(lower=1e-300)
    df["pval"] = df["pval"].clip(lower=1e-300)

    # GSEA ranking metric
    #I used pval instead of padj because padj is too conservative and may lead to missing important genes
    df["rank"] = ( np.sign(df["log2FC"]) * -np.log10(df["pval"])) 
    ranking = (df[["gene","rank"]].sort_values("rank", ascending=False ))

    return ranking

# Cnet plot function

In [35]:
def make_cnetplot(gsea_result, output, top_n=5):

    res = gsea_result.copy()

    # Significant pathways
    res = res[ res["FDR q-val"] < 0.05]

    if len(res) == 0:
        return

    # top pathways by NES magnitude
    res["absNES"] = abs(res["NES"])

    pathways = (res.sort_values("absNES", ascending=False).head(top_n))
    G = nx.Graph()
    for _, row in pathways.iterrows():
        pathway = row["Term"]

        # leading edge genes
        genes = row["Lead_genes"]

        if pd.isna(genes):
            continue

        genes = genes.split(";")
        G.add_node(pathway, type="pathway")

        for gene in genes:
            G.add_node(gene, type="gene")
            G.add_edge(pathway, gene)

    if len(G.nodes)==0:
        return

    plt.figure(figsize=(12,10))

    pos = nx.spring_layout(G, seed=42 )

    pathway_nodes = [
        n for n,d in G.nodes(data=True)
        if d["type"]=="pathway"
    ]

    gene_nodes = [
        n for n,d in G.nodes(data=True)
        if d["type"]=="gene"
    ]

    nx.draw_networkx_nodes(G, pos, nodelist=pathway_nodes, node_size=1500)
    nx.draw_networkx_nodes(G, pos, nodelist=gene_nodes, node_size=300)
    nx.draw_networkx_edges(G, pos, alpha=0.4)
    nx.draw_networkx_labels(G, pos, font_size=8)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(output, dpi=300, bbox_inches="tight")

    plt.close()


# Run GSEA

In [36]:
for file in files:

    print("\nRunning:", file)
    path = os.path.join( input_dir, file)

    # read DE results
    de = pd.read_csv(path)
    ranking = create_rank_file(de)
    comparison = (file.replace(".csv",""))
    rank_file = os.path.join(output_dir, comparison+"_ranking.rnk")
    ranking.to_csv(rank_file, sep="\t", index=False, header=False)

    for db_name, database in gene_sets.items():
        print("  ", db_name)
        outdir = os.path.join(output_dir, comparison, db_name)
        os.makedirs(outdir, exist_ok=True)

        try:
            prerank = gp.prerank(
                rnk=ranking,
                gene_sets=database,
                processes=4,
                permutation_num=1000,
                min_size=15,
                max_size=500,
                outdir=outdir,
                seed=42,
                verbose=False
            )
            results = prerank.res2d

            results.to_csv(os.path.join(outdir, "GSEA_results.csv"))

            # ----------------------------
            # CNET plot
            # ----------------------------
            
            cnet_file = os.path.join(outdir, "cnetplot.png")
            make_cnetplot(results, cnet_file, top_n=5)

            # ----------------------------
            # GSEA dotplot
            # ----------------------------

            gp.dotplot(
                results,
                column="FDR q-val",
                title=f"{comparison} {db_name}",
                cutoff=0.25,
                size=10,
                figsize=(8,6),
                ofname=os.path.join(outdir, "dotplot.png")
            )

        except Exception as e:
            print("FAILED:", db_name, e)

print("\nFinished")


Running: DE_AAD_vs_AC_Aged_AD_vs_Aged_Control.csv
   Hallmark
FAILED: Hallmark Warning: No enrich terms when cutoff = 0.25
   GO_BP


/tmp/ipykernel_2509327/4097624994.py:19: DeprecationWarning: processes is deprecated; use threads
  prerank = gp.prerank(
/tmp/ipykernel_2509327/4097624994.py:19: DeprecationWarning: processes is deprecated; use threads
  prerank = gp.prerank(


   Reactome


/tmp/ipykernel_2509327/4097624994.py:19: DeprecationWarning: processes is deprecated; use threads
  prerank = gp.prerank(
/tmp/ipykernel_2509327/4097624994.py:19: DeprecationWarning: processes is deprecated; use threads
  prerank = gp.prerank(
2026-07-16 16:35:51,314 [ERROR] No supported gene_sets: KEGG_2021_Mouse


   KEGG
FAILED: KEGG Error parsing gmt parameter for gene sets
   WikiPathways


...

   Reactome


/tmp/ipykernel_2509327/4097624994.py:19: DeprecationWarning: processes is deprecated; use threads
  prerank = gp.prerank(
/tmp/ipykernel_2509327/4097624994.py:19: DeprecationWarning: processes is deprecated; use threads
  prerank = gp.prerank(
2026-07-16 16:36:49,764 [ERROR] No supported gene_sets: KEGG_2021_Mouse


   KEGG
FAILED: KEGG Error parsing gmt parameter for gene sets
   WikiPathways


/tmp/ipykernel_2509327/4097624994.py:19: DeprecationWarning: processes is deprecated; use threads
  prerank = gp.prerank(
2026-07-16 16:36:53,120 [ERROR] No supported gene_sets: DisGeNET_2024


   DisGeNET
FAILED: DisGeNET Error parsing gmt parameter for gene sets

Finished
